In [ ]:
# dim_customers = spark.read.csv("work/output/silver/customers.csv", header=True, inferSchema=True) \
#     .select("customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state") \
#     .dropDuplicates(["customer_id"])

# # dim_sellers
# dim_sellers = spark.read.csv("work/output/silver/sellers.csv", header=True, inferSchema=True) \
#     .select("seller_id", "seller_zip_code_prefix", "seller_city", "seller_state") \
#     .dropDuplicates(["seller_id"])

# # dim_products
# dim_products = spark.read.csv("work/output/silver/products.csv", header=True, inferSchema=True) \
#     .select("product_id", "product_category_name", "product_name_lenght", "product_description_lenght") \
#     .dropDuplicates(["product_id"])


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Load Silver Tables
customers = spark.read.csv("work/output/silver/customers.csv", header=True, inferSchema=True)
orders = spark.read.csv("work/output/silver/orders.csv", header=True, inferSchema=True)
items = spark.read.csv("work/output/silver/order_items.csv", header=True, inferSchema=True)

# 2. Join Customers with Orders and Items to get spend/dates
# We join items to orders first to get the price per order_id
order_spend = items.withColumn("item_total", F.col("price") + F.col("freight_value")) \
                   .groupBy("order_id") \
                   .agg(F.sum("item_total").alias("order_total"))

customer_orders = orders.join(order_spend, on="order_id", how="left") \
                        .join(customers, on="customer_id", how="inner")

# 3. Use Window Function to find the "Most Recent" record per customer_unique_id
window_spec = Window.partitionBy("customer_unique_id").orderBy(F.col("order_purchase_timestamp").desc())

# 4. Create the Dimension with Metrics
dim_customers = (customer_orders
    .withColumn("row_num", F.row_number().over(window_spec))
    .groupBy("customer_unique_id")
    .agg(
        # Location from the most recent order row
        F.first(F.when(F.col("row_num") == 1, F.col("customer_city"))).alias("customer_city"),
        F.first(F.when(F.col("row_num") == 1, F.col("customer_state"))).alias("customer_state"),
        F.first(F.when(F.col("row_num") == 1, F.col("customer_zip_code_prefix").cast("string"))).alias("customer_zip_code_prefix"),
        # Lifecycle Metrics
        F.min(F.to_date("order_purchase_timestamp")).alias("first_order_date"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("order_total").cast("decimal(18,2)").alias("total_spend")
    )
    # Add Boolean Flag
    .withColumn("is_repeat_customer", F.col("total_orders") > 1)
    # Add Surrogate Key (Monotonically increasing ID)
    .withColumn("customer_key", F.monotonically_increasing_id())
)

# 5. Save to Gold
dim_customers.write.mode("overwrite").csv("work/output/gold/dim_customers", header=True)

In [5]:
# 1. Load Silver Tables
products = spark.read.csv("work/output/silver/products.csv", header=True, inferSchema=True)

In [6]:
products.columns

['product_category_name',
 'product_description_lenght',
 'product_height_cm',
 'product_id',
 'product_length_cm',
 'product_name_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_width_cm',
 '_ingested_at',
 '_source_endpoint',
 '_is_valid']

In [9]:
dim_products = (products
.withColumn("product_key", F.monotonically_increasing_id())
.withColumn("product_category_name", when(F.col("product_category_name").isNull(), F.lit("UNKNOWN")).otherwise(F.col("product_category_name")))
.withColumn("product_volume_cm3",when(F.col("product_length_cm").isNull() | F.col("product_height_cm").isNull() | F.col("product_width_cm").isNull(), None)
.otherwise(F.col("product_width_cm") * F.col("product_height_cm") * F.col("product_length_cm"))).select("product_key", "product_id", "product_category_name", "product_weight_g", "product_volume_cm3", "product_photos_qty","product_description_lenght") 
)

In [10]:
# 5. Save to Gold
dim_products.write.mode("overwrite").csv("work/output/gold/dim_products", header=True)

In [ ]:
seller_key	integer	Surrogate key
seller_id	string	sellers
seller_city	string	sellers
seller_state	string	sellers
seller_zip_code_prefix	string	sellers

In [11]:
sellers = spark.read.csv("work/output/silver/sellers.csv", header=True, inferSchema=True)

In [13]:
sellers = sellers.withColumn("seller_key",  F.monotonically_increasing_id()).select("seller_key", "seller_id"
, "seller_city", "seller_state", "seller_zip_code_prefix")
                                                                                    
sellers.write.mode("overwrite").csv("work/output/gold/dim_products", header=True)

In [15]:
# 1. Load Silver Tables
payments = spark.read.csv("work/output/silver/payments.csv", header=True, inferSchema=True)


In [18]:
# 2. Process Payments (Highest value type and Max installments per Order)
pay_window = Window.partitionBy("order_id").orderBy(F.desc("payment_value"), F.asc("payment_type"))

order_payments = payments.withColumn("rn", F.row_number().over(pay_window)) \
    .groupBy("order_id") \
    .agg(
        F.sum("payment_value").alias("total_order_payment"),
        F.first(F.when(F.col("rn") == 1, F.col("payment_type"))).alias("payment_type"),
        F.max("payment_installments").alias("payment_installments")
    )

In [19]:
order_totals = items.groupBy("order_id").agg(F.sum("price").alias("total_items_price"))

items_with_payment = items.join(order_totals, on="order_id") \
    .join(order_payments, on="order_id") \
    .withColumn("item_share", F.col("price") / F.col("total_items_price")) \
    .withColumn("distributed_payment", (F.col("item_share") * F.col("total_order_payment")).cast("decimal(18,2)"))


In [23]:
items_with_payment.join(orders, on="order_id").columns

['order_id',
 'freight_value',
 'order_item_id',
 'price',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 '_ingested_at',
 '_source_endpoint',
 '_is_valid',
 'total_items_price',
 'total_order_payment',
 'payment_type',
 'payment_installments',
 'item_share',
 'distributed_payment',
 'customer_id',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_purchase_timestamp',
 'order_status',
 '_ingested_at',
 '_source_endpoint',
 '_is_valid']

In [29]:
fact_order_items = items_with_payment.join(orders, on="order_id")


In [31]:
fact_order_items.columns

['order_id',
 'freight_value',
 'order_item_id',
 'price',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 '_ingested_at',
 '_source_endpoint',
 '_is_valid',
 'total_items_price',
 'total_order_payment',
 'payment_type',
 'payment_installments',
 'item_share',
 'distributed_payment',
 'customer_id',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_purchase_timestamp',
 'order_status',
 '_ingested_at',
 '_source_endpoint',
 '_is_valid']

In [32]:
customers_silver = spark.read.csv("work/output/silver/customers.csv", header=True, inferSchema=True) \
    .select("customer_id", "customer_unique_id")

fact_order_items = fact_order_items.join(customers_silver, on="customer_id", how="left")

fact_order_items = fact_order_items.join(dim_customers, on="customer_unique_id", how="left") \
    .select(
        F.monotonically_increasing_id().alias("order_item_sk"),
        "order_id",
        "order_item_id",
        "customer_key", 
        "product_id",   
        "seller_id",    
        F.to_date("order_purchase_timestamp").alias("order_date"),
        "order_status",
        F.col("price").cast("decimal(18,2)"),
        F.col("freight_value").cast("decimal(18,2)"),
        "distributed_payment",
        "payment_type",
        "payment_installments",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp").alias("days_to_deliver"),
        F.datediff("order_delivered_customer_date", "order_estimated_delivery_date").alias("days_delivery_vs_estimate")
    ) \
    .withColumn("is_late_delivery", F.col("days_delivery_vs_estimate") > 0)

In [34]:
fact_order_items.write.mode("overwrite").csv("work/output/gold/fact_order_items", header=True)